In [1]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler, PowerTransformer, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from pathlib import Path

In [2]:
import dagshub
dagshub.init(repo_owner='AMR-ITH', repo_name='RealEstateInsights', mlflow=True)
import mlflow

# set the tracking server

mlflow.set_tracking_uri("https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow")

# mlflow experiment

mlflow.set_experiment("Exp 3 - RF-HP Tuning-30")

Accessing as AMR-ITH

Initialized MLflow to track repo "AMR-ITH/RealEstateInsights"

Repository AMR-ITH/RealEstateInsights initialized!

<Experiment: artifact_location='mlflow-artifacts:/977cb1ccd3cd4a8198e4876b6a083c11', creation_time=1746527225080, experiment_id='5', last_update_time=1746527225080, lifecycle_stage='active', name='Exp 3 - RF-HP Tuning-30', tags={}>

In [3]:
# pathlib is a module in Python that provides an object-oriented interface 
# for working with file system paths.
current = Path.cwd()
parent = current.parent

# load the dat 
df = pd.read_csv(parent /'data/interim_data.csv')
df.head()

df.drop(columns=['carpet_area','super_bulit_area','nearbylocation','facility','apartment_name','appartment_loc'], inplace=True)

In [4]:
temp_df = df.copy()

X = temp_df.drop(columns=['price_value'])
y = temp_df['price_value']

In [5]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [6]:
# do the basic processing input data

num_cols = ['bulit_area','luxury_facility_scores']
nomial_cols = ['zone']
ordinal_cols = ['construction_status','bhk_type']


In [7]:
bhk_type_order = ['1','2','3','4','5','6','7','8','9','10']
construction_status_order = ['New Property','Under Construction', 'Relatively New', 'Moderatly Old', 'Old','undefined']

In [8]:
# build a preprocessor

prepocessor = ColumnTransformer(transformers=[
    ("scale", MinMaxScaler(), num_cols),
        ("nominal_encode", OneHotEncoder(handle_unknown="ignore",sparse_output=False), nomial_cols),
    ("ordinal_encode", OrdinalEncoder(categories=[construction_status_order,bhk_type_order]), ordinal_cols)
],remainder="passthrough",n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False)

prepocessor.set_output(transform="pandas")

ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                  remainder='passthrough',
                  transformers=[('scale', MinMaxScaler(),
                                 ['bulit_area', 'luxury_facility_scores']),
                                ('nominal_encode',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['zone']),
                                ('ordinal_encode',
                                 OrdinalEncoder(categories=[['New Property',
                                                             'Under '
                                                             'Construction',
                                                             'Relatively New',
                                                             'Moderatly Old',
                                                             'Old',
                                                             'undefined'],
                                                            ['1', '2', '3', '4',
                                                             '5', '6', '7', '8',
                                                             '9', '10']]),
                                 ['construction_status', 'bhk_type'])],
                  verbose_feature_names_out=False)

In [9]:
# transform the data

X_train_trans = prepocessor.fit_transform(X_train)
X_test_trans = prepocessor.transform(X_test)

X_train_trans

,bulit_area,luxury_facility_scores,zone_east,zone_north,zone_south,zone_west,construction_status,bhk_type
2842,0.197885,0.313901,0.0,1.0,0.0,0.0,0.0,2.0
903,0.185153,0.488789,1.0,0.0,0.0,0.0,2.0,2.0
3262,0.297475,0.412556,0.0,1.0,0.0,0.0,2.0,2.0
109,0.114804,0.322870,1.0,0.0,0.0,0.0,1.0,1.0
5602,0.092361,0.000000,0.0,0.0,0.0,1.0,5.0,1.0
...,...,...,...,...,...,...,...,...
3772,0.224860,0.390135,0.0,1.0,0.0,0.0,0.0,3.0
5191,0.140160,0.295964,0.0,0.0,1.0,0.0,0.0,2.0
5226,0.175874,0.573991,0.0,0.0,1.0,0.0,4.0,2.0
5390,0.118041,0.309417,0.0,0.0,0.0,1.0,2.0,1.0


In [13]:
from sklearn.ensemble import RandomForestRegressor
import optuna

from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import cross_val_score

In [11]:
def objective(trial):
    with mlflow.start_run(nested=True):
        params = {
            "n_estimators": trial.suggest_int("n_estimators",10,500),
            "max_depth": trial.suggest_int("max_depth",1,30),
            "max_features": trial.suggest_categorical("max_features",[None,"sqrt","log2"]),
            "min_samples_split": trial.suggest_int("min_samples_split",2,10),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf",1,10),
            "max_samples": trial.suggest_float("max_samples",0.5,1),
            "random_state": 42,
            "n_jobs": -1,
        }

        # log model parameters
        mlflow.log_params(params)

        # build the model
        rf = RandomForestRegressor(**params)

        # train the model
        rf.fit(X_train_trans,y_train)

        # get the predictions
        y_pred_train = rf.predict(X_train_trans)
        y_pred_test = rf.predict(X_test_trans)


        # perform cross validation
        cv_score = cross_val_score(rf,
                                X_train_trans,
                                y_train,
                                cv=5,
                                n_jobs=-1)

        # mean score
        mean_score = cv_score.mean()

        # log avg cross val error
        mlflow.log_metric("cross_val_error",mean_score)

        return mean_score

In [ ]:
def objective(trial):
    with mlflow.start_run(nested=True):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 10, 500),
            "max_depth": trial.suggest_int("max_depth", 1, 30),
            "max_features": trial.suggest_categorical("max_features", [None, "sqrt", "log2"]),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "max_samples": trial.suggest_float("max_samples", 0.5, 1),
            "bootstrap": trial.suggest_categorical("bootstrap", [True]),  # Always True if using max_samples
            "random_state": 42,
            "n_jobs": -1,
        }

        # Log hyperparameters
        mlflow.log_params(params)

        # Build and train the model
        rf = RandomForestRegressor(**params)
        rf.fit(X_train_trans, y_train)

        # Predictions
        y_pred_train = rf.predict(X_train_trans)
        y_pred_test = rf.predict(X_test_trans)

        # Basic evaluation
        mae_test = mean_absolute_error(y_test, y_pred_test)
        r2_test = r2_score(y_test, y_pred_test)

        # Cross-validation on train data
        cv_mae_scores_train = cross_val_score(
            rf, X_train_trans, y_train, cv=5,
            scoring="neg_mean_absolute_error", n_jobs=-1
        )
        cv_r2_scores_train = cross_val_score(
            rf, X_train_trans, y_train, cv=5,
            scoring="r2", n_jobs=-1
        )

        # Cross-validation on test data (optional, but included for insight)
        cv_mae_scores_test = cross_val_score(
            rf, X_test_trans, y_test, cv=5,
            scoring="neg_mean_absolute_error", n_jobs=-1
        )
        cv_r2_scores_test = cross_val_score(
            rf, X_test_trans, y_test, cv=5,
            scoring="r2", n_jobs=-1
        )

        # Compute means
        mean_cv_mae_train = -cv_mae_scores_train.mean()
        mean_cv_r2_train = cv_r2_scores_train.mean()
        mean_cv_mae_test = -cv_mae_scores_test.mean()
        mean_cv_r2_test = cv_r2_scores_test.mean()

        # Log metrics
        mlflow.log_metric("cv_mae_train", mean_cv_mae_train)
        mlflow.log_metric("cv_r2_train", mean_cv_r2_train)
        mlflow.log_metric("cv_mae_test", mean_cv_mae_test)
        mlflow.log_metric("cv_r2_test", mean_cv_r2_test)
        mlflow.log_metric("test_mae", mae_test)
        mlflow.log_metric("test_r2", r2_test)

        # Print for tracking
        print("train mae:", mean_cv_mae_train)
        print("train r2:", mean_cv_r2_train)
        print("test mae:", mean_cv_mae_test)
        print("test r2:", mean_cv_r2_test)

        # Store Optuna user attributes
        trial.set_user_attr("mae_test", mae_test)
        trial.set_user_attr("r2_test", r2_test)
        trial.set_user_attr("cv_mae_train", mean_cv_mae_train)
        trial.set_user_attr("cv_r2_train", mean_cv_r2_train)
        trial.set_user_attr("cv_mae_test", mean_cv_mae_test)
        trial.set_user_attr("cv_r2_test", mean_cv_r2_test)

        # Final score to minimize
        combined_score = mae_test / (1 + max(0, r2_test))
        mlflow.log_metric("combined_score", combined_score)

        return combined_score


In [14]:
# create optuna study
study = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="best_model"):
    # optimize the objective function
    study.optimize(objective,n_trials=30,n_jobs=-1,show_progress_bar=True)

    # log the best parameters
    mlflow.log_params(study.best_params)

    # log the best score
    mlflow.log_metric("best_score",study.best_value)

    # train the model on best parameters
    best_rf = RandomForestRegressor(**study.best_params)

    best_rf.fit(X_train_trans,y_train)

    y_pred_train = best_rf.predict(X_train_trans)
    y_pred_test = best_rf.predict(X_test_trans)
    mlflow.sklearn.log_model(best_rf,"model")
    mlflow.log_params(best_rf.get_params())


    scores = cross_val_score(best_rf,X_train_trans,y_train,cv=5,n_jobs=-1)

    # mae,r2 for test and train
    mae_train = mean_absolute_error(y_train,y_pred_train)
    r2_train = r2_score(y_train,y_pred_train)
    mae_test = mean_absolute_error(y_test,y_pred_test)
    r2_test = r2_score(y_test,y_pred_test)

    mlflow.log_metric("mae-train",mae_train)
    mlflow.log_metric("r2-train",r2_train)
    mlflow.log_metric("mae-test",mae_test)
    mlflow.log_metric("r2-test",r2_test)

        # log the best model
    mlflow.sklearn.log_model(best_rf,artifact_path="model")


[I 2025-05-08 10:25:59,674] A new study created in memory with name: no-name-2722a4e4-3f7c-436f-be51-b79a627d828f
  0%|          | 0/30 [00:00<?, ?it/s]

🏃 View run clumsy-shrew-801 at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5/runs/8c45049557be424993594b9f14ebb48d
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5
🏃 View run bedecked-worm-913 at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5/runs/45ef7a64d7794c17b1b4c412dc4ff118
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5


Best trial: 4. Best value: 0.223152:   3%|▎         | 1/30 [00:29<14:29, 29.98s/it]

[I 2025-05-08 10:26:30,374] Trial 4 finished with value: 0.2231524582495708 and parameters: {'n_estimators': 219, 'max_depth': 22, 'max_features': None, 'min_samples_split': 5, 'min_samples_leaf': 10, 'max_samples': 0.796880627152603, 'bootstrap': True}. Best is trial 4 with value: 0.2231524582495708.


Best trial: 4. Best value: 0.223152:   7%|▋         | 2/30 [00:34<07:07, 15.26s/it]

...

[I 2025-05-08 10:30:19,395] Trial 29 finished with value: 0.21212801710407078 and parameters: {'n_estimators': 474, 'max_depth': 27, 'max_features': 'sqrt', 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_samples': 0.7049300091634824, 'bootstrap': True}. Best is trial 16 with value: 0.21079419637833222.


Best trial: 16. Best value: 0.210794: 100%|██████████| 30/30 [04:20<00:00,  8.67s/it]


[I 2025-05-08 10:30:20,401] Trial 28 finished with value: 0.21095274296338903 and parameters: {'n_estimators': 484, 'max_depth': 30, 'max_features': 'sqrt', 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_samples': 0.709964768440107, 'bootstrap': True}. Best is trial 16 with value: 0.21079419637833222.


2025/05/08 10:30:41 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/05/08 10:40:26 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run best_model at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5/runs/e35d860ef6db4d74a4238095514557d0
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5


In [17]:
study.best_params

{'n_estimators': 478,
 'max_depth': 30,
 'max_features': 'sqrt',
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_samples': 0.7048412420826664,
 'bootstrap': True}

In [50]:
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
import time

In [24]:


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 10.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 10.0),
        "random_state": 42,
        "n_jobs": -1,
    }
    
    # build the model
    lgbm = LGBMRegressor(**params)
    
    # train the model
    lgbm.fit(X_train_trans, y_train)
    
    # get the predictions
    y_pred_train = lgbm.predict(X_train_trans)
    y_pred_test = lgbm.predict(X_test_trans)

    
    # Calculate metrics
    mae_test = mean_absolute_error(y_test, y_pred_test)
    r2_test = r2_score(y_test, y_pred_test)

    # perform cross validation for MAE
    cv_mae_scores_train = cross_val_score(
        lgbm,
        X_train_trans,
        y_train,
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )
    
    cv_mae_scores_test = cross_val_score(
        lgbm,
        X_test_trans,
        y_test,
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )
    
    # perform cross validation for R2
    cv_r2_scores_train = cross_val_score(
        lgbm,
        X_train_trans,
        y_train,
        cv=5,
        scoring="r2",
        n_jobs=-1
    )

    cv_r2_scores_test = cross_val_score(
        lgbm,
        X_test_trans,
        y_test,
        cv=5,
        scoring="r2",
        n_jobs=-1
    )
    
    
    # mean scores from cross-validation train
    mean_cv_mae_train = -cv_mae_scores_train.mean()
    mean_cv_r2_train = cv_r2_scores_train.mean()

    mean_cv_mae_test = -cv_mae_scores_test.mean()
    mean_cv_r2_test = cv_r2_scores_test.mean() 
    print("train mae",mean_cv_mae_train)
    print("train r2",mean_cv_r2_train)

    print("test mae",mean_cv_mae_test)
    print("test r2",mean_cv_r2_test)   
    
    # Store metrics in trial user attributes for display
    trial.set_user_attr("mae_test", mae_test)
    trial.set_user_attr("r2_test", r2_test)
    trial.set_user_attr("cv_mae_train", mean_cv_mae_train)
    trial.set_user_attr("cv_r2_train", mean_cv_r2_train)
    trial.set_user_attr("cv_mae_test",mean_cv_mae_test)
    trial.set_user_attr("cv_mae_test",mean_cv_r2_test)
    
    # Combined score - we want to minimize MAE and maximize R2
    # Lower value is better
    combined_score = mae_test / (1 + max(0, r2_test))
    
    return combined_score



In [27]:
# optimize the objective function
study.optimize(objective, n_trials=30, n_jobs=-1, show_progress_bar=True)






Best trial: 17. Best value: 0.250019:   3%|▎         | 1/30 [00:35<17:07, 35.41s/it]

train mae 0.5179493878254922
train r2 0.7919859618000679
test mae 0.5687820078189965
test r2 0.7438664323126739
[I 2025-05-08 17:50:05,940] Trial 43 finished with value: 0.27331670053544543 and parameters: {'n_estimators': 182, 'max_depth': 36, 'learning_rate': 0.08623651551553785, 'num_leaves': 58, 'min_child_samples': 92, 'subsample': 0.5023564016642542, 'colsample_bytree': 0.7816565741300607, 'reg_alpha': 7.6098046169437215, 'reg_lambda': 9.07841121226063}. Best is trial 17 with value: 0.2500185449653238.


Best trial: 17. Best value: 0.250019:   7%|▋         | 2/30 [00:36<07:00, 15.03s/it]

train mae 0.5129678565698365
train r2 0.7974721649444667
test mae 0.553117137699578
test r2 0.7620264981756553
[I 2025-05-08 17:50:06,715] Trial 48 finished with value: 0.26722258747500877 and parameters: {'n_estimators': 186, 'max_depth': 37, 'learning_rate': 0.15293223632802871, 'num_leaves': 111, 'min_child_samples': 49, 'subsample': 0.5645163953925569, 'colsample_bytree': 0.7865228480056493, 'reg_alpha': 9.250632056947799, 'reg_lambda': 4.172941880283828}. Best is trial 17 with value: 0.2500185449653238.


Best trial: 17. Best value: 0.250019:  10%|█         | 3/30 [00:37<04:00,  8.90s/it]

...

train mae 0.5119955714918782
train r2 0.7993199813282864
test mae 0.5511627540937183
test r2 0.7721677775158866
[I 2025-05-08 17:52:16,929] Trial 67 finished with value: 0.26547161555709603 and parameters: {'n_estimators': 223, 'max_depth': 14, 'learning_rate': 0.2558616645602224, 'num_leaves': 38, 'min_child_samples': 28, 'subsample': 0.7265719988169806, 'colsample_bytree': 0.660789513692106, 'reg_alpha': 7.595394877398239, 'reg_lambda': 7.238217330380452}. Best is trial 17 with value: 0.2500185449653238.


Best trial: 17. Best value: 0.250019:  93%|█████████▎| 28/30 [02:46<00:02,  1.01s/it]

train mae 0.5129033894240589
train r2 0.7996123303271319
test mae 0.5517374797729705
test r2 0.7698413911712352
[I 2025-05-08 17:52:17,340] Trial 68 finished with value: 0.26785785107737636 and parameters: {'n_estimators': 237, 'max_depth': 14, 'learning_rate': 0.23079551976639556, 'num_leaves': 21, 'min_child_samples': 27, 'subsample': 0.7693401402078017, 'colsample_bytree': 0.6586737811824726, 'reg_alpha': 7.6963017286613695, 'reg_lambda': 6.806496310417832}. Best is trial 17 with value: 0.2500185449653238.
train mae 0.5072023165036408
train r2 0.7991239366872576
test mae 0.5487592362475564
test r2 0.7687400537561336
[I 2025-05-08 17:52:17,508] Trial 69 finished with value: 0.26754319197246074 and parameters: {'n_estimators': 152, 'max_depth': 14, 'learning_rate': 0.23695331193503777, 'num_leaves': 125, 'min_child_samples': 30, 'subsample': 0.7396294112062634, 'colsample_bytree': 0.9898738663577338, 'reg_alpha': 8.380933379749257, 'reg_lambda': 2.6596428774273475}. Best is trial 17 w

Best trial: 17. Best value: 0.250019: 100%|██████████| 30/30 [02:47<00:00,  5.58s/it]

train mae 0.5186736349126139
train r2 0.7908219947254413
test mae 0.5745016021240866
test r2 0.7384776716590205
[I 2025-05-08 17:52:17,610] Trial 70 finished with value: 0.27305202538528855 and parameters: {'n_estimators': 144, 'max_depth': 14, 'learning_rate': 0.24966539755716194, 'num_leaves': 42, 'min_child_samples': 100, 'subsample': 0.7358594973106954, 'colsample_bytree': 0.6487520187942994, 'reg_alpha': 8.39196852645544, 'reg_lambda': 2.608606897298494}. Best is trial 17 with value: 0.2500185449653238.
train mae 0.5209396945602824
train r2 0.7913015933121736
test mae 0.5775696247957247
test r2 0.7375239281893241
[I 2025-05-08 17:52:17,782] Trial 71 finished with value: 0.27555232657176415 and parameters: {'n_estimators': 149, 'max_depth': 21, 'learning_rate': 0.24081678333638917, 'num_leaves': 44, 'min_child_samples': 100, 'subsample': 0.7439673762853298, 'colsample_bytree': 0.6871293612447112, 'reg_alpha': 9.89176992007244, 'reg_lambda': 3.1397824907765957}. Best is trial 17 wit

In [28]:
# train the model on best parameters
best_rf = LGBMRegressor(**study.best_params)

best_rf.fit(X_train_trans,y_train)

y_pred_train = best_rf.predict(X_train_trans)
y_pred_test = best_rf.predict(X_test_trans)



scores = cross_val_score(best_rf,X_train_trans,y_train,cv=5,n_jobs=-1)

# mae,r2 for test and train
mae_train = mean_absolute_error(y_train,y_pred_train)
r2_train = r2_score(y_train,y_pred_train)
mae_test = mean_absolute_error(y_test,y_pred_test)
r2_test = r2_score(y_test,y_pred_test)


print(mae_train)
print(r2_train)
print(mae_test)
print(r2_test)

[LightGBM] [Warning] Unknown parameter: max_samples
[LightGBM] [Warning] Unknown parameter: max_features
[LightGBM] [Warning] Unknown parameter: min_samples_split
[LightGBM] [Warning] min_data_in_leaf is set with min_child_samples=20, will be overridden by min_samples_leaf=1. Current value: min_data_in_leaf=1
[LightGBM] [Warning] Unknown parameter: bootstrap
[LightGBM] [Warning] Unknown parameter: max_samples
[LightGBM] [Warning] Unknown parameter: max_features
[LightGBM] [Warning] Unknown parameter: min_samples_split
[LightGBM] [Warning] min_data_in_leaf is set with min_child_samples=20, will be overridden by min_samples_leaf=1. Current value: min_data_in_leaf=1
[LightGBM] [Warning] Unknown parameter: bootstrap
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000371 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 470
[LightGBM] [Info] Number of data points in the train set: 4860, number of used feat

In [18]:
study.best_params

{'n_estimators': 150,
 'max_depth': 25,
 'learning_rate': 0.22571315117777171,
 'num_leaves': 94,
 'min_child_samples': 48,
 'subsample': 0.7859608525116539,
 'colsample_bytree': 0.6713298072955733,
 'reg_alpha': 2.9441365937007995,
 'reg_lambda': 8.073108740850945}

In [17]:
def objective(trial):
    with mlflow.start_run(nested=True):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 10, 500),
            "max_depth": trial.suggest_int("max_depth", 1, 30),
            "max_features": trial.suggest_categorical("max_features", [None, "sqrt", "log2"]),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "max_samples": trial.suggest_float("max_samples", 0.5, 1),
            "bootstrap": trial.suggest_categorical("bootstrap", [True]),  # Always True if using max_samples
            "random_state": 42,
            "n_jobs": -1,
        }

        # Log hyperparameters
        mlflow.log_params(params)

        # Build and train the model
        rf = RandomForestRegressor(**params)
        rf.fit(X_train_trans, y_train)

        # Predictions
        y_pred_train = rf.predict(X_train_trans)
        y_pred_test = rf.predict(X_test_trans)

        # Basic evaluation
        mae_test = mean_absolute_error(y_test, y_pred_test)
        r2_test = r2_score(y_test, y_pred_test)

        # Cross-validation on train data
        cv_mae_scores_train = cross_val_score(
            rf, X_train_trans, y_train, cv=5,
            scoring="neg_mean_absolute_error", n_jobs=-1
        )
        cv_r2_scores_train = cross_val_score(
            rf, X_train_trans, y_train, cv=5,
            scoring="r2", n_jobs=-1
        )

        # Cross-validation on test data (optional, but included for insight)
        cv_mae_scores_test = cross_val_score(
            rf, X_test_trans, y_test, cv=5,
            scoring="neg_mean_absolute_error", n_jobs=-1
        )
        cv_r2_scores_test = cross_val_score(
            rf, X_test_trans, y_test, cv=5,
            scoring="r2", n_jobs=-1
        )

        # Compute means
        mean_cv_mae_train = -cv_mae_scores_train.mean()
        mean_cv_r2_train = cv_r2_scores_train.mean()
        mean_cv_mae_test = -cv_mae_scores_test.mean()
        mean_cv_r2_test = cv_r2_scores_test.mean()

        # Log metrics
        mlflow.log_metric("cv_mae_train", mean_cv_mae_train)
        mlflow.log_metric("cv_r2_train", mean_cv_r2_train)
        mlflow.log_metric("cv_mae_test", mean_cv_mae_test)
        mlflow.log_metric("cv_r2_test", mean_cv_r2_test)
        mlflow.log_metric("test_mae", mae_test)
        mlflow.log_metric("test_r2", r2_test)

        # Print for tracking
        print("train mae:", mean_cv_mae_train)
        print("train r2:", mean_cv_r2_train)
        print("test mae:", mean_cv_mae_test)
        print("test r2:", mean_cv_r2_test)

        # Store Optuna user attributes
        trial.set_user_attr("mae_test", mae_test)
        trial.set_user_attr("r2_test", r2_test)
        trial.set_user_attr("cv_mae_train", mean_cv_mae_train)
        trial.set_user_attr("cv_r2_train", mean_cv_r2_train)
        trial.set_user_attr("cv_mae_test", mean_cv_mae_test)
        trial.set_user_attr("cv_r2_test", mean_cv_r2_test)

        # Final score to minimize
        # combined_score = mae_test / (1 + max(0, r2_test))
        # mlflow.log_metric("combined_score", combined_score)

        return mean_cv_mae_test


In [11]:
import optuna

c:\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
# import mlflow

study = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="best_model"):
    # optimize the objective function
    study.optimize(objective,n_trials=30,n_jobs=-1,show_progress_bar=True)

    # log the best parameters
    mlflow.log_params(study.best_params)

    # log the best score
    mlflow.log_metric("best_score",study.best_value)

    # train the model on best parameters
    best_rf = RandomForestRegressor(**study.best_params)

    best_rf.fit(X_train_trans,y_train)

    y_pred_train = best_rf.predict(X_train_trans)
    y_pred_test = best_rf.predict(X_test_trans)
    mlflow.sklearn.log_model(best_rf,"model")
    mlflow.log_params(best_rf.get_params())


    scores = cross_val_score(best_rf,X_train_trans,y_train,cv=5,n_jobs=-1)

    # mae,r2 for test and train
    mae_train = mean_absolute_error(y_train,y_pred_train)
    r2_train = r2_score(y_train,y_pred_train)
    mae_test = mean_absolute_error(y_test,y_pred_test)
    r2_test = r2_score(y_test,y_pred_test)

    mlflow.log_metric("mae-train",mae_train)
    mlflow.log_metric("r2-train",r2_train)
    mlflow.log_metric("mae-test",mae_test)
    mlflow.log_metric("r2-test",r2_test)

        # log the best model
    mlflow.sklearn.log_model(best_rf,artifact_path="model")

[I 2025-05-09 09:32:19,767] A new study created in memory with name: no-name-f2f6955e-a504-44da-8156-e27aa0caff55
  0%|          | 0/30 [00:00<?, ?it/s]

train mae: 0.5103936565948618
train r2: 0.7956767939861777
test mae: 0.5477688259770757
test r2: 0.7652423674187258
train mae: 0.6919350171033666
train r2: 0.6700926004193033
test mae: 0.6800774432436081
test r2: 0.6796899651871771
train mae: 0.4923158141685483
train r2: 0.8061874158579773
test mae: 0.5336332583396923
test r2: 0.7770930477207129
train mae: 0.8302960978063219
train r2: 0.5737250960281174
test mae: 0.7746237166857874
test r2: 0.6155418545216982
🏃 View run nebulous-zebra-823 at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5/runs/e40bee65f5644961af0b056ed4195cf9
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5
🏃 View run monumental-tern-493 at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5/runs/52ce15a292c44910854b4030f321ea41
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5
🏃 View run nimble-cat-723 at: https://dagshub.com/AMR-ITH/RealEstat

Best trial: 7. Best value: 0.547769:   3%|▎         | 1/30 [01:00<29:27, 60.94s/it]

[I 2025-05-09 09:33:21,649] Trial 7 finished with value: 0.5477688259770757 and parameters: {'n_estimators': 120, 'max_depth': 24, 'max_features': None, 'min_samples_split': 4, 'min_samples_leaf': 9, 'max_samples': 0.7263455350930672, 'bootstrap': True}. Best is trial 7 with value: 0.5477688259770757.


Best trial: 7. Best value: 0.547769:   7%|▋         | 2/30 [01:03<12:31, 26.84s/it]

...

[I 2025-05-09 09:37:07,651] Trial 26 finished with value: 0.5346979521749398 and parameters: {'n_estimators': 50, 'max_depth': 26, 'max_features': 'log2', 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_samples': 0.8558354119678249, 'bootstrap': True}. Best is trial 22 with value: 0.5208613943109931.
train mae: 0.4886406829774906
train r2: 0.8108524316161972
test mae: 0.5285295226490441
test r2: 0.7842409438769944
🏃 View run nebulous-frog-342 at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5/runs/dbc46d9b528d43828f23b536515cd52d
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5


Best trial: 22. Best value: 0.520861: 100%|██████████| 30/30 [04:49<00:00,  9.66s/it]


[I 2025-05-09 09:37:10,639] Trial 29 finished with value: 0.5285295226490441 and parameters: {'n_estimators': 286, 'max_depth': 24, 'max_features': 'log2', 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_samples': 0.8304688621065146, 'bootstrap': True}. Best is trial 22 with value: 0.5208613943109931.


2025/05/09 09:37:33 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/05/09 09:37:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run best_model at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5/runs/8118406ba243491f82577bb46f58ad7d
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5


In [19]:
study.best_params

{'n_estimators': 10,
 'max_depth': 20,
 'max_features': 'log2',
 'min_samples_split': 7,
 'min_samples_leaf': 1,
 'max_samples': 0.8878905282972512,
 'bootstrap': True}

In [20]:
# train the model on best parameters
best_rf = RandomForestRegressor(**study.best_params)

best_rf.fit(X_train_trans,y_train)

y_pred_train = best_rf.predict(X_train_trans)
y_pred_test = best_rf.predict(X_test_trans)



scores = cross_val_score(best_rf,X_train_trans,y_train,cv=5,n_jobs=-1)

# mae,r2 for test and train
mae_train = mean_absolute_error(y_train,y_pred_train)
r2_train = r2_score(y_train,y_pred_train)
mae_test = mean_absolute_error(y_test,y_pred_test)
r2_test = r2_score(y_test,y_pred_test)


print(mae_train)
print(r2_train)
print(mae_test)
print(r2_test)

0.3277480879137463
0.9119286098769239
0.4768208403594557
0.8216717264382034
